# Saving and loading in JSON

A `Laminate` saved with `save_json` is loaded identical by `load_json`, from a
file name or a file object, see [Saving and loading](../saving_loading.rst).
The function `make_laminate` of the tests creates a laminate with
`laminated_plate`. The code below is the unit test `test_save_load` of
`tests/test_json.py`, with a temporary folder as `tmp_path`.

In [1]:
import sys
sys.path.insert(0, '../tests') # helpers of the tests

import numpy as np

import io
import json
import tempfile
from pathlib import Path

from composites import (laminated_plate, to_dict, from_dict, save_json,
                        load_json)
from test_pickle import assert_same, make_laminate, laminaprop

tmp_path = Path(tempfile.mkdtemp())

In [2]:
lam = make_laminate()
fname = tmp_path / 'laminate.json'
save_json(lam, fname)
assert_same(lam, load_json(fname))
assert_same(lam, load_json(str(fname)))
with open(fname) as f:
    assert json.load(f)['type'] == 'Laminate'
buf = io.StringIO()
save_json(lam, buf, indent=None)
assert '\n' not in buf.getvalue()
buf.seek(0)
assert_same(lam, load_json(buf))
print(fname.read_text()[:600], '...')

{
  "type": "Laminate",
  "format_version": 2,
  "composites_version": "0.10.3",
  "data": {
    "A11": 334080598.7520903,
    "A12": 101402190.69173962,
    "A16": 42653765.74751955,
    "A22": 401052289.2496578,
    "A26": 15345419.557762777,
    "A66": 117315645.63078767,
    "B11": -213716.37386114942,
    "B12": -3941.6202559449302,
    "B16": 73148.56912001516,
    "B22": 221599.61437303916,
    "B26": 4877.703645623187,
    "B66": -3941.6202559449302,
    "D11": 1586.616685442299,
    "D12": 277.92910370225275,
    "D16": 337.1122068985246,
    "D22": 671.3369153088771,
    "D26": 164.1 ...


## Embedding a laminate in another JSON document

`to_dict` and `from_dict` embed a laminate in another JSON document (unit test
`test_example_embed_in_json_document` of `tests/test_json.py`):

In [3]:
lam = laminated_plate([0, 45, -45, 90], plyt=0.125e-3,
                      laminaprop=laminaprop, rho=1600.)
# to_dict returns JSON types, to be embedded in another JSON document
doc = {'name': 'skin', 'mesh_size': 0.01, 'laminate': to_dict(lam)}
fname = tmp_path / 'model.json'
with open(fname, 'w') as f:
    json.dump(doc, f)
with open(fname) as f:
    doc2 = json.load(f)
lam2 = from_dict(doc2['laminate'])
np.testing.assert_array_equal(lam2.ABD, lam.ABD)
np.testing.assert_array_equal(lam2.Ats, lam.Ats)
assert lam2.intrhoz6 == lam.intrhoz6
print(sorted(doc2))
print(sorted(doc2['laminate']))

['laminate', 'mesh_size', 'name']
['composites_version', 'data', 'format_version', 'type']
